In [0]:
df_active_cc_members = spark.sql("""
  SELECT *
  FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
  WHERE snapshot_date_id = DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 2), 'yyyyMMdd')
    AND renewal_status LIKE 'Active%'
""")

# print(f"Active CineClub Members (yesterday's snapshot): {df_active_cc_members.count():,}")
# display(df_active_cc_members)

In [0]:
%pip install openpyxl -q
import pandas as pd

xlsx_path = "/Volumes/data_exploration_hub/marketing_hub/offline_data/General Admit Prices.xlsx"
df_ga_xlsx = pd.read_excel(xlsx_path, sheet_name="Sheet3")

# Melt wide (one col per year) -> long, keeping LocationId + year + price
year_cols = [c for c in df_ga_xlsx.columns if c != "LocationID"]
df_ga_long = (
    df_ga_xlsx
    .melt(id_vars="LocationID", value_vars=year_cols, var_name="visit_year", value_name="ga_ticket_price")
    .rename(columns={"LocationID": "LocationId"})
)
df_ga_long["visit_year"] = df_ga_long["visit_year"].astype(int)

# Register as Spark temp view — join will match on LocationId AND year of visit
spark.createDataFrame(df_ga_long).createOrReplaceTempView("ga_prices_excel")
print(f"GA prices loaded: {len(df_ga_long)} rows ({df_ga_long['LocationId'].nunique()} locations x {len(year_cols)} years)")
display(df_ga_long.head(10))

In [0]:
# Register active CC members as temp view for joining
df_active_cc_members.createOrReplaceTempView("active_cc_members")

df_cc_transactions = spark.sql("""
  WITH title_watched AS (
    -- Only keep titles where net ticket quantity > 0 (actually watched, not fully refunded)
    SELECT 
      t.CDE_ID,
      t.VisitDateId,
      t.LocationId,
      f.title_id
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    INNER JOIN active_cc_members m
      ON CAST(t.CDE_ID AS STRING) = m.cde_id
    LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_film f
      ON t.film_id = f.film_id
    WHERE t.CineClubFLAG = 'Yes'
      AND t.TicketFLAG = 'Yes'
    GROUP BY t.CDE_ID, t.VisitDateId, t.LocationId, f.title_id
    HAVING SUM(t.Quantity) > 0
  ),
  title_count_per_visit AS (
    SELECT CDE_ID, VisitDateId, LocationId, COUNT(DISTINCT title_id) AS distinct_title_count
    FROM title_watched
    GROUP BY CDE_ID, VisitDateId, LocationId
  )
  SELECT 
    t.CDE_ID,
    t.VisitDateId,
    t.LocationId,
    SUM(CASE WHEN TicketFLAG = 'Yes' THEN Net_Revenue ELSE 0 END) AS ticket_revenue,
    SUM(CASE WHEN TicketFLAG = 'Yes' THEN Quantity ELSE 0 END) AS ticket_count,
    SUM(CASE WHEN ConcessionFLAG = 'Yes' AND CineClub_Discount_Type_ID = 2 THEN Net_Revenue ELSE 0 END) AS discounted_concession_revenue,
    SUM(CASE WHEN ConcessionFLAG = 'Yes' AND CineClub_Discount_Type_ID = 3 THEN Net_Revenue ELSE 0 END) AS discounted_gaming_revenue,
    MAX(CASE WHEN TicketFLAG = 'Yes' THEN 1 ELSE 0 END) AS is_ticket_visit,
    MAX(CASE WHEN ConcessionFLAG = 'Yes' THEN 1 ELSE 0 END) AS is_concession_visit,
    COALESCE(MAX(tc.distinct_title_count), 0) AS distinct_title_count,
    MAX(CASE WHEN COTFLAG = 'Yes' THEN 1 ELSE 0 END) AS is_online_booking,
    SUM(CASE WHEN CineClub_Discount_Type_ID = 1 THEN Quantity ELSE 0 END) AS discount_type_1_count,
    SUM(CASE WHEN CineClub_Discount_Type_ID = 2 THEN Quantity ELSE 0 END) AS discount_type_2_count,
    SUM(CASE WHEN CineClub_Discount_Type_ID = 3 THEN Quantity ELSE 0 END) AS discount_type_3_count,
    SUM(CASE WHEN CineClub_Discount_Type_ID = 5 THEN Quantity ELSE 0 END) AS discount_type_5_count,
    SUM(CASE WHEN CineClub_Discount_Type_ID = 6 THEN Quantity ELSE 0 END) AS discount_type_6_count,
    MAX(p.ga_ticket_price) AS ga_ticket_price
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
  INNER JOIN active_cc_members m
    ON CAST(t.CDE_ID AS STRING) = m.cde_id
  LEFT JOIN title_count_per_visit tc
    ON t.CDE_ID = tc.CDE_ID AND t.VisitDateId = tc.VisitDateId AND t.LocationId = tc.LocationId
  LEFT JOIN ga_prices_excel p
    ON t.LocationId = p.LocationId
    AND YEAR(TO_DATE(CAST(t.VisitDateId AS STRING), 'yyyyMMdd')) = p.visit_year
  WHERE t.CineClubFLAG = 'Yes'
  GROUP BY t.CDE_ID, t.VisitDateId, t.LocationId, t.film_id
""")
# df_cc_transactions.cache()
print(f"CC Member Visits: {df_cc_transactions.count():,}")

In [0]:
df_cc_transactions.display()

In [0]:
# Register visit-level DataFrame as temp view
df_cc_transactions.createOrReplaceTempView("cc_transactions")

df_cc_member_savings = spark.sql("""
  WITH savings AS (
    SELECT
      *,
      ROUND(COALESCE(discount_type_5_count + discount_type_6_count, 0) * (COALESCE(ga_ticket_price, 0) - 9.99), 2) AS ticket_savings,
      CASE WHEN ticket_savings = 0 THEN 0 ELSE is_online_booking END AS Online_Booking_Count,
      ROUND(COALESCE(discounted_concession_revenue, 0) * 0.25, 2) AS concession_savings,
      ROUND(COALESCE(discounted_gaming_revenue, 0) * 0.25, 2) AS gaming_savings,
      ROUND( ticket_savings + concession_savings + gaming_savings + Online_Booking_Count,2 ) AS total_savings
    FROM cc_transactions
  )
  SELECT
    CDE_ID,
    COUNT(*)                             AS total_visits,
    SUM(is_ticket_visit)                 AS total_ticket_visits,
    SUM(is_concession_visit)             AS total_concession_visits,
    SUM(ticket_revenue)                  AS ticket_revenue,
    SUM(ticket_count)                    AS ticket_count,
    ROUND(SUM(discounted_concession_revenue), 2)   AS discounted_concession_revenue,
    ROUND(SUM(discounted_gaming_revenue), 2)       AS discounted_gaming_revenue,
    ROUND(SUM(distinct_title_count), 2)            AS distinct_title_count,
    ROUND(SUM(discount_type_1_count), 2)           AS discount_type_1_count,
    ROUND(SUM(discount_type_2_count), 2)           AS discount_type_2_count,
    ROUND(SUM(discount_type_3_count), 2)           AS discount_type_3_count,
    ROUND(SUM(discount_type_5_count), 2)           AS discount_type_5_count,
    ROUND(SUM(discount_type_6_count), 2)           AS discount_type_6_count,
    ROUND(SUM(Online_Booking_Count), 2)            AS online_booking_savings,
    ROUND(SUM(ticket_savings), 2)        AS ticket_savings,
    ROUND(SUM(concession_savings), 2)    AS concession_savings,
    ROUND(SUM(gaming_savings), 2)        AS gaming_savings,
    ROUND(SUM(total_savings), 2)         AS total_savings
  FROM savings
  GROUP BY CDE_ID
""")

print(f"CC Members with savings data: {df_cc_member_savings.count():,}")
display(df_cc_member_savings)